# Frozen Unity player lifecycle

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
"""Construct a frozen native player plus verified adapter; record every decision."""
from pathlib import Path
from types import SimpleNamespace
import os,json,time,shutil,hashlib
import numpy as np
import gymnasium as gym
from contract import ROOT,CODE,verify_inputs,write_json
from environment_adapter import MatchedBaseline,estimator_state_hash

class Runtime:
    def __init__(self,config,stage_dir,simulator_seed):
        self.config=config;self.directory=Path(stage_dir);self.seed=simulator_seed
        self.env=self.unity=self.process=None;self.handshake=[]
        self.inputs_before=verify_inputs();self.before_hash=None
    def create(self):
        from affectively.environments import create_environment,GymToGymnasiumWrapper
        from affectively.environments import base as base_module
        from affectively.models.linear_model import LinearSurrogateModel
        from mlagents_envs.environment import UnityEnvironment
        from mlagents_envs.rpc_communicator import RpcCommunicator
        build=self.directory/'runtime_build'
        shutil.copytree(CODE/'affectively/builds/solid/Linux',build)
        executable=build/'solid.x86_64'
        original_launch=base_module.BaseEnvironment.load_environment
        original_init=RpcCommunicator.initialize
        original_train=LinearSurrogateModel.train_model
        def no_refit(model):raise RuntimeError('Missing frozen estimator: refitting forbidden')
        def capture(comm,inputs,*args,**kwargs):
            self.handshake.append(int(inputs.rl_initialization_input.seed))
            return original_init(comm,inputs,*args,**kwargs)
        def launch(native,identifier,graphics,args):
            self.unity=UnityEnvironment(str(executable),worker_id=identifier,seed=self.seed,no_graphics=True,
                timeout_wait=20,side_channels=[native.engineConfigChannel,native.customSideChannel],
                additional_args=args,log_folder=str(self.directory))
            self.process=self.unity._process
            write_json(self.directory/'process.json',{'runner_pid':os.getpid(),'unity_pid':self.process.pid,'worker_id':identifier,'port':5005+identifier})
            return self.unity
        base_module.BaseEnvironment.load_environment=launch;RpcCommunicator.initialize=capture
        LinearSurrogateModel.train_model=no_refit
        try:
            args=SimpleNamespace(run=self.config['worker_id'],game='solid',headless=1,weight=.5,cluster=0,
                target_arousal=1,periodic_ra=0,cv=0,grayscale=0,discretize=0,classifier=1,preference=1,decision_period=10,imitate=0)
            native=create_environment(args,args.run);native.simulator_initialization_seed=self.seed
            self.env=MatchedBaseline(GymToGymnasiumWrapper(native),self.config['condition'])
        finally:
            base_module.BaseEnvironment.load_environment=original_launch;RpcCommunicator.initialize=original_init
            LinearSurrogateModel.train_model=original_train
        if self.handshake!=[self.seed]:raise RuntimeError('Unity initialization seed mismatch')
        self.before_hash=estimator_state_hash(native.model)
        self.recorder=Recorder(self.env,self.directory)
        return self.recorder
    def close(self):
        outcome={};errors=[]
        try:
            if self.env is not None:self.env.score_model.assert_frozen()
        except BaseException as exc:errors.append('preclose freeze: '+repr(exc))
        try:
            if hasattr(self,'recorder'):
                self.recorder.finish()
                if hasattr(self,'monitor'):self.monitor.close()
                else:self.recorder.close()
            elif self.env is not None:self.env.close()
        except BaseException as exc:errors.append('wrapper close: '+repr(exc))
        try:
            if self.process is not None and self.process.poll() is None and self.unity is not None:
                outcome['fallback_close_required']=True;self.unity.close()
        except BaseException as exc:errors.append('raw Unity close: '+repr(exc))
        outcome['unity_exited']=self.process is None or self.process.poll() is not None
        try:
            if self.env is not None:
                outcome['estimator_state_before']=self.before_hash
                outcome['estimator_state_after']=estimator_state_hash(self.env.env.env.model)
                if outcome['estimator_state_before']!=outcome['estimator_state_after']:errors.append('Estimator state drift')
            outcome['initialization_handshake_seeds']=self.handshake
            outcome['static_inputs_preserved']=verify_inputs()==self.inputs_before
            timer=self.directory/'runtime_build/solid_Data/ML-Agents/Timers/Gym Solid Rally_timers.json'
            if self.process is not None and timer.exists():shutil.copy2(timer,self.directory/'unity_timers.json')
        except BaseException as exc:errors.append('postclose preservation: '+repr(exc))
        outcome['errors']=errors;write_json(self.directory/'runtime_result.json',outcome)
        if errors or not outcome['unity_exited'] or not outcome.get('static_inputs_preserved') or outcome.get('fallback_close_required'):
            raise RuntimeError('Runtime cleanup/freeze check failed: '+repr(outcome))
        return outcome


from telemetry_recording import Recorder
UnityRuntime=Runtime
print('Frozen Unity player lifecycle definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Frozen increasing-preference scoring definitions/execution completed.
Task progress and fresh-pair rewards definitions/execution completed.
Matched observation action and window adapter definitions/execution completed.
Decision and episode telemetry definitions/execution completed.
Frozen Unity player lifecycle definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
from contract import verify_inputs
print('Verified static inputs:',len(verify_inputs()),'; launch occurs only through an explicit experiment run.')

Verified static inputs: 280 ; launch occurs only through an explicit experiment run.
